# Script 4 — Participant-level questionnaire master file

Reduces the participant-day file from script 3 to one row per participant,
applies the NHANES missing-value codes to the questionnaire items the
analyses use, and writes the questionnaire master file that scripts 4_5 and
5 read.

1. Aggregates the accelerometer day-level MIMS total (`PAXMTSD`) to one row
   per participant.
2. Cleans the PHQ-9 items and computes the total score.
3. Cleans sleep duration and the two sleep-problem items.
4. Cleans the cognitive sub-tests and computes the CERAD memory total.
5. Computes self-reported activity minutes per day from the PAQ items.
6. Recodes sex to 0/1 and the education refusal / don't-know codes to
   missing, and merges everything into the master file.

## Input

- `data/processed/0_check_personlevel.csv` — script 3.

## Output

- `data/processed/5_paday_dp_sl_cf_paq.csv` — one row per participant
  (19,931 rows). The intermediate files `paday_raw.csv`, `1_paday.csv`,
  `2_paday_dp.csv`, `3_paday_dp_sl.csv`, `4_paday_dp_sl_cf.csv` and
  `paq_minutes_per_day.csv` are written to `data/processed/` on the way and
  are read by no other script.

## Cleaning rules

- PHQ-9 items `DPQ010`–`DPQ090`: codes 7 (refused), 9 (don't know) and `.`
  set to missing. `dpq_sum` requires all nine items (`min_count = 9`);
  `depression` = 1 if `dpq_sum` ≥ 10.
- Sleep duration `SLD010H`: codes 77 and 99 set to missing; values below 2 h
  set to missing; values above 12 h set to 12 (NHANES top-codes at 12, so
  this rule changes no value in these cycles). `SLQ050` and `SLQ060`:
  codes 7 and 9 set to missing, 2 recoded to 0.
- Cognitive sub-tests: `.` set to missing. The CERAD memory total
  `memory_score` requires all four trials (`min_count = 4`); `fluency_score`
  = `CFDAST`; `speed_score` = `CFDDS`. Scores of zero are treated as
  not administered in script 9, not here.
- Sex `RIAGENDR`: 1 = male, 2 = female recoded to 0 = male, 1 = female.
- Education `DMDEDUC2`: five-level ordinal scale kept as is; codes 7 and 9
  set to missing.
- Self-reported activity: for each PAQ domain, minutes per day = days per
  week × minutes per day / 7 when the gate item is "yes", 0 when it is "no";
  the domain sum is capped at 960 min/day. These variables enter no model;
  they are carried in the master file only.

## Step 0 — Paths

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd


def find_root():
    """Locate the project root without assuming where the kernel started."""
    env = os.environ.get('NHANES_SEM_ROOT')
    cands = ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().parents]
    for q in cands:
        if (q / 'data').is_dir() and (q / 'scripts').is_dir():
            return q
    raise RuntimeError('cannot find the project root from ' + str(Path.cwd())
                       + ' -- start Jupyter inside the project, or set NHANES_SEM_ROOT')


BASE = find_root()
PROC = BASE / 'data' / 'processed'
print('project root:', BASE)

## Step 1 — One row per participant from the accelerometer day-level file

In [ ]:
df = pd.read_csv(PROC / '0_check_personlevel.csv')

# Day-level rows with a PAXMTSD value
pa_day = df[['SEQN', 'PAXMTSD']].dropna(subset=['PAXMTSD'])

# Per-participant mean, max, min, std and count of the daily MIMS total
pa_feature = (
    pa_day.groupby('SEQN')
    .agg(pa_day_mean=('PAXMTSD', 'mean'),
         pa_day_max=('PAXMTSD', 'max'),
         pa_day_min=('PAXMTSD', 'min'),
         pa_day_std=('PAXMTSD', 'std'),
         pa_day_count=('PAXMTSD', 'count'))
    .reset_index()
)
# pa_valid = 1 with at least 3 days carrying a daily total
pa_feature['pa_valid'] = pa_feature['pa_day_count'].apply(lambda x: 1 if x >= 3 else 0)
pa_feature.to_csv(PROC / 'paday_raw.csv', index=False)
print(f'participants with a daily MIMS total: {len(pa_feature):,}')

# Merge back onto one row per participant (re-read from disk, as every
# later step re-reads its input)
master = pd.read_csv(PROC / '0_check_personlevel.csv')
pa_feature = pd.read_csv(PROC / 'paday_raw.csv')
master_person = master.drop_duplicates(subset=['SEQN'])
out = pd.merge(master_person, pa_feature, on='SEQN', how='left')
out.to_csv(PROC / '1_paday.csv', index=False)
print(f'wrote 1_paday.csv: {len(out):,} participants')

## Step 2 — PHQ-9 items and total

In [ ]:
df = pd.read_csv(PROC / '1_paday.csv')
dpq_cols = ['DPQ010', 'DPQ020', 'DPQ030', 'DPQ040', 'DPQ050', 'DPQ060', 'DPQ070', 'DPQ080', 'DPQ090']

# Codes 7 (refused), 9 (don't know) and '.' -> missing
for col in dpq_cols:
    df[col] = df[col].replace([7, 9, '.'], pd.NA)
    df[col] = pd.to_numeric(df[col], errors='coerce')

# The total requires all nine items (min_count=9): a missing item gives a
# missing total, not a partial sum. Same definition as PHQ9_total in
# scripts 6, 8 and 9.
df['dpq_sum'] = df[dpq_cols].sum(axis=1, min_count=9)

# 1 if PHQ-9 >= 10, 0 if below, missing if the total is missing
df['depression'] = (df['dpq_sum'] >= 10).astype('Int64')
df.loc[df['dpq_sum'].isna(), 'depression'] = pd.NA

df.to_csv(PROC / '2_paday_dp.csv', index=False)
print(f'PHQ-9 total observed for {int(df["dpq_sum"].notna().sum()):,} participants; '
      f'range {df["dpq_sum"].min():.0f}-{df["dpq_sum"].max():.0f}')
print('depression flag:', df['depression'].value_counts(dropna=False).to_dict())

## Step 3 — Sleep duration and sleep-problem items

In [ ]:
df = pd.read_csv(PROC / '2_paday_dp.csv')

# SLD010H, usual weekday sleep, hours: codes 77, 99 and '.' -> missing
df['SLD010H'] = df['SLD010H'].replace([77, 99, '.'], pd.NA)
df['SLD010H'] = pd.to_numeric(df['SLD010H'], errors='coerce')
# Valid range 2-12 h: below 2 -> missing; above 12 -> 12
df.loc[df['SLD010H'] < 2, 'SLD010H'] = pd.NA
df.loc[df['SLD010H'] > 12, 'SLD010H'] = 12

# SLQ050 (told a doctor about trouble sleeping) and SLQ060 (told a doctor
# of a sleep disorder): codes 7, 9, '.' -> missing; 1 = yes, 2 -> 0 = no
for col in ['SLQ050', 'SLQ060']:
    df[col] = df[col].replace([7, 9, '.'], pd.NA)
    df[col] = df[col].replace(2, 0)
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df.loc[~df[col].isin([0, 1]) & df[col].notna(), col] = pd.NA

# Named copies
df['sleep_hours'] = df['SLD010H']
df['sleep_trouble'] = df['SLQ050']
df['sleep_disorder'] = df['SLQ060']

df.to_csv(PROC / '3_paday_dp_sl.csv', index=False)
print(f'sleep duration observed for {int(df["sleep_hours"].notna().sum()):,} participants; '
      f'mean {df["sleep_hours"].mean():.2f} h, range {df["sleep_hours"].min():.0f}-{df["sleep_hours"].max():.0f}')
print('sleep trouble :', df['sleep_trouble'].value_counts(dropna=False).to_dict())
print('sleep disorder:', df['sleep_disorder'].value_counts(dropna=False).to_dict())

## Step 4 — Cognitive sub-tests

In [ ]:
df = pd.read_csv(PROC / '3_paday_dp_sl.csv')

# CERAD Word List: three learning trials and delayed recall (CFDCST1-3, CFDCSR);
# Animal Fluency (CFDAST); Digit Symbol Substitution (CFDDS)
cf_cols = ['CFDCST1', 'CFDCST2', 'CFDCST3', 'CFDCSR', 'CFDAST', 'CFDDS']
for col in cf_cols:
    df[col] = df[col].replace(['.'], pd.NA)
    df[col] = pd.to_numeric(df[col], errors='coerce')

# The memory total requires all four trials (min_count=4), so a missing
# trial gives a missing total rather than a spurious low score.
df['memory_score'] = df[['CFDCST1', 'CFDCST2', 'CFDCST3', 'CFDCSR']].sum(axis=1, min_count=4)
df['fluency_score'] = df['CFDAST']
df['speed_score'] = df['CFDDS']

df.to_csv(PROC / '4_paday_dp_sl_cf.csv', index=False)
for col in ['memory_score', 'fluency_score', 'speed_score']:
    s = df[col]
    print(f'{col:14s} observed {int(s.notna().sum()):5,}   mean {s.mean():6.2f}   range {s.min():.0f}-{s.max():.0f}')

## Step 5 — Self-reported activity minutes per day from the PAQ items

These variables are carried in the master file and enter no model.

In [ ]:
raw_df = pd.read_csv(PROC / '4_paday_dp_sl_cf.csv')

# PAQ domains: gate item, days-per-week item, minutes-per-day item
domain_map = {
    'vig_work':  {'gate': 'PAQ605', 'days': 'PAQ610', 'mins': 'PAD615'},
    'mod_work':  {'gate': 'PAQ620', 'days': 'PAQ625', 'mins': 'PAD630'},
    'transport': {'gate': 'PAQ635', 'days': 'PAQ640', 'mins': 'PAD645'},
    'vig_rec':   {'gate': 'PAQ650', 'days': 'PAQ655', 'mins': 'PAD660'},
    'mod_rec':   {'gate': 'PAQ665', 'days': 'PAQ670', 'mins': 'PAD675'},
}


def clean_numeric(series, missing_codes, lower=None, upper=None):
    """Missing codes -> NaN; values outside [lower, upper] -> NaN."""
    s = series.replace(missing_codes, np.nan)
    s = pd.to_numeric(s, errors='coerce')
    if lower is not None:
        s = s.mask(s < lower, np.nan)
    if upper is not None:
        s = s.mask(s > upper, np.nan)
    return s


def process_paq(df_in, domain_upper_limit, total_day_cap=960, suffix='main'):
    """Minutes per day in each PAQ domain, their sum (capped), and a strict
    version requiring all five domains complete."""
    df = df_in.copy()
    avg_cols, complete_flags = [], []
    for domain, cols in domain_map.items():
        g, d, m = cols['gate'], cols['days'], cols['mins']
        # Missing codes -> NaN; days 0-7; minutes 10 to domain_upper_limit
        df[g] = clean_numeric(df[g], [7, 9, '.'], lower=1, upper=2)
        df[d] = clean_numeric(df[d], [77, 99, '.'], lower=0, upper=7)
        df[m] = clean_numeric(df[m], [7777, 9999, '.'], lower=10, upper=domain_upper_limit)

        avg_col, flag_col = f'{domain}_avg_{suffix}', f'{domain}_ok_{suffix}'
        df[avg_col], df[flag_col] = np.nan, 0
        # Gate = 2 (no) -> 0 min/day
        no_mask = (df[g] == 2)
        df.loc[no_mask, avg_col] = 0
        df.loc[no_mask, flag_col] = 1
        # Gate = 1 (yes) with days >= 1 and minutes present -> days x minutes / 7
        ok_mask = (df[g] == 1) & df[d].notna() & (df[d] >= 1) & df[m].notna()
        df.loc[ok_mask, avg_col] = (df[d] * df[m]) / 7.0
        df.loc[ok_mask, flag_col] = 1
        avg_cols.append(avg_col)
        complete_flags.append(flag_col)

    raw_sum = df[avg_cols].sum(axis=1, min_count=1)
    df[f'sub_mvpa_rawsum_{suffix}'] = raw_sum
    df[f'sub_mvpa_{suffix}'] = raw_sum.clip(upper=total_day_cap)
    df[f'n_comp_{suffix}'] = df[complete_flags].sum(axis=1)
    df[f'sub_mvpa_strict_{suffix}'] = np.where(df[f'n_comp_{suffix}'] == 5,
                                               raw_sum.clip(upper=total_day_cap), np.nan)
    return df[['SEQN', f'sub_mvpa_{suffix}', f'sub_mvpa_rawsum_{suffix}',
               f'sub_mvpa_strict_{suffix}', f'n_comp_{suffix}']]


# Main version (domain limit 900 min/day) and a stricter one (720 min/day)
df_main = process_paq(raw_df, 900, suffix='main')
df_sens = process_paq(raw_df, 720, suffix='sens')
final = pd.merge(df_main, df_sens, on='SEQN')
# Sitting time PAD680: codes 7777, 9999 -> NaN; 0-1440 min/day
final['sit_min_day'] = clean_numeric(raw_df['PAD680'], [7777, 9999, '.'], lower=0, upper=1440)
final.to_csv(PROC / 'paq_minutes_per_day.csv', index=False)

print(final[['sub_mvpa_main', 'sub_mvpa_strict_main', 'sit_min_day']]
      .agg(['mean', 'std', 'median', 'max']).round(1).to_string())
print(f'participants capped at 960 min/day: {int((final["sub_mvpa_rawsum_main"] > 960).sum())}')

## Step 6 — Sex and education recodes; write the master file

In [ ]:
master = pd.read_csv(PROC / '4_paday_dp_sl_cf.csv')
pa_min = pd.read_csv(PROC / 'paq_minutes_per_day.csv')

# Sex: NHANES 1 = male, 2 = female -> 0 = male, 1 = female
master['RIAGENDR'] = master['RIAGENDR'].replace({1: 0, 2: 1})

# Education: DMDEDUC2 is a five-level ordinal scale (1 = less than 9th grade
# to 5 = college graduate); codes 7 (refused) and 9 (don't know) -> missing
n_edu_codes = int(master['DMDEDUC2'].isin([7, 9]).sum())
master['DMDEDUC2'] = pd.to_numeric(master['DMDEDUC2'].replace({7: np.nan, 9: np.nan}), errors='coerce')
print(f'education: {n_edu_codes} values of 7 or 9 set to missing')

m1 = master.merge(pa_min, on='SEQN', how='left')
out = PROC / '5_paday_dp_sl_cf_paq.csv'
m1.to_csv(out, index=False)
print(f'wrote {out.name}: {len(m1):,} rows x {m1.shape[1]} columns; '
      f'{m1["SEQN"].nunique():,} participants')